<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_77_gpu_networking_nvlink_ib_roce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🛜 **Module 77 — GPU Networking (NVLink · InfiniBand · RoCE)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps


# Module 77 — GPU Networking

> M66 said "all-reduce gradients across GPUs." This module is **how that line actually happens**. Below your `torch.distributed` call sits a stack: **NCCL → CUDA IPC → NVLink (within a node) → RDMA over InfiniBand or RoCE (between nodes)**. Get any layer wrong and your 256-GPU training run becomes a 100-GPU training run with the other 156 GPUs eating coffee. Get them right and you hit the **70-80 % of theoretical peak FLOPs** that frontier labs report.
>
> This is the network half of distributed training. There's no Colab demo — you can't fake a 400 Gb/s fabric — but every command and tuning knob here is exactly what you type in production.

### What you'll cover
1. The hierarchy — why "all-reduce" is a multi-layer problem
2. **NCCL** — the collective library underneath every framework
3. **NVLink + NVSwitch** — intra-node GPU↔GPU
4. **PCIe** — the slow fallback bus
5. **InfiniBand** — the gold-standard inter-node fabric
6. **RoCE v2** — RDMA over Ethernet, the cheaper alternative
7. **GPUDirect RDMA** — GPU memory → wire without a CPU hop
8. **Rail-optimised topology** — how DGX / HGX racks are wired
9. **Tuning NCCL** — env vars, `nccl-tests`, `nvidia-smi topo -m`
10. The 2025-26 fabric landscape — NVLink 5, NDR / XDR IB, Spectrum-X


## 1 · Why all-reduce is a stack

```
   Python                  loss.backward(); optimizer.step()
        │
        ▼
   PyTorch / JAX        torch.distributed → all_reduce
        │
        ▼
   NCCL                 ring-/tree-/double-binary-tree algorithm
        │
        ├──── INTRA-NODE ──── NVLink ◀── NVSwitch fabric ───── GPU 0 .. GPU 7
        │
        ▼
   INTER-NODE
        ├── GPUDirect RDMA → InfiniBand HCA → IB switch → fat-tree
        │   (or)
        └── GPUDirect RDMA → ConnectX RoCE NIC → RoCE-capable Ethernet
```

Every name above maps to a physical thing or a kernel module on the box. **The fabric is not abstracted away** — when training slows down you fix it here, not in your model code.

## 2 · NCCL — the collective library

**NVIDIA Collective Communications Library**. Every framework that does multi-GPU training calls into NCCL underneath: PyTorch DDP / FSDP, JAX, DeepSpeed, Megatron, vLLM (M44), TensorRT-LLM.

| Collective | What it does | Where it shows up |
|---|---|---|
| **AllReduce** | sum (or avg) a tensor across all ranks; everyone gets the result | DDP gradient sync (M66 §3) |
| **AllGather** | every rank concatenates everyone's shard into the full tensor | FSDP unsharding (M66 §4) |
| **ReduceScatter** | reduce across ranks, then keep only a shard on each | FSDP gradient sharding |
| **Broadcast** | one rank's tensor sent to all | initial weights spread |
| **AlltoAll** | each rank sends a *different* slice to each other rank | Mixture-of-Experts routing; sequence-parallel |
| **Send/Recv** | point-to-point | pipeline-parallel boundary passes |

NCCL picks the **algorithm** for each collective based on:
- the **topology** it discovers at startup (`NCCL_DEBUG=INFO` prints it),
- the **buffer size**,
- the **available bandwidth** at each link.

For AllReduce on N GPUs in a ring, NCCL does **`2(N-1)/N`** of the buffer's bytes per link — independent of N. That's why ring all-reduce *scales*.

## 3 · NVLink + NVSwitch — intra-node

When two GPUs in the same server need to talk, they go over **NVLink** — NVIDIA's point-to-point GPU interconnect.

| Generation | Per-link | Per-GPU aggregate | Used by |
|---|---|---|---|
| NVLink 3 | 50 GB/s | 600 GB/s | A100 |
| NVLink 4 | 100 GB/s | 900 GB/s | H100 |
| NVLink 5 | ~200 GB/s | 1.8 TB/s | B100 / B200 / Blackwell |

In an 8-GPU server **NVSwitch** turns the GPUs into a fully-connected mesh: every GPU sees the full per-link bandwidth to every other GPU, simultaneously. That's why an 8× H100 box is the unit of frontier-LLM purchases — within the box, you're bandwidth-rich.

> 🔑 **Check what you have:** `nvidia-smi topo -m` prints a matrix of pairwise GPU connectivity. Look for `NV12` (12 NVLinks), `NV4`, `PIX` (PCIe via switch), `SYS` (across NUMA — slow). If you see `SYS` between two GPUs you wanted to talk fast — fix it.

In [ ]:
# What the output of `nvidia-smi topo -m` looks like on an 8x H100 box.
example_topo = '''
        GPU0   GPU1   GPU2   GPU3   GPU4   GPU5   GPU6   GPU7    CPU Affinity
GPU0     X     NV18   NV18   NV18   NV18   NV18   NV18   NV18    0-47
GPU1    NV18    X     NV18   NV18   NV18   NV18   NV18   NV18    0-47
GPU2    NV18   NV18    X     NV18   NV18   NV18   NV18   NV18    0-47
...
Legend:
  X    = Self
  NV#  = Connection via that many NVLinks (NV18 = 18 NVLinks)
  PIX  = Through a PCIe switch
  PHB  = PCIe host bridge
  SYS  = Across NUMA nodes / sockets — SLOW
'''
print(example_topo)

## 4 · PCIe — the slow fallback

If two GPUs don't share NVLink (or you're on a workstation), they fall back to **PCIe**:

| PCIe gen | Per-lane | x16 lane GPU |
|---|---|---|
| Gen 3 | 1 GB/s | 16 GB/s |
| Gen 4 | 2 GB/s | 32 GB/s |
| Gen 5 | 4 GB/s | 64 GB/s |
| Gen 6 | 8 GB/s | 128 GB/s |

Even Gen 5 x16 is **~14× slower** than NVLink 5. If you see PCIe in your `nvidia-smi topo`, your all-reduce is bandwidth-bound at that link — and any tensor parallelism you try will crawl.

> 🟡 **NUMA matters.** On a dual-socket server, each socket owns half the PCIe lanes. A GPU on socket 0 talking to a NIC on socket 1 traverses the inter-socket link — slow and worth checking with `numactl` / `lstopo`.

## 5 · InfiniBand — the gold-standard inter-node fabric

When the model doesn't fit on one node (M66 §6 — tensor + pipeline), GPUs on different machines need to talk. **InfiniBand (IB)** is the dominant fabric.

| Generation | Per-port | Notes |
|---|---|---|
| HDR | 200 Gb/s | A100-era |
| **NDR** | **400 Gb/s** | **H100 reference** |
| XDR | 800 Gb/s | B100 / B200 / GB200 reference; 2024+ |

Three properties that make IB right for ML training:
1. **RDMA** — remote-DMA. The sending GPU's NIC writes **directly** into the receiving GPU's memory without involving either CPU. Microsecond latencies, near-zero CPU overhead.
2. **Lossless** — switches use credit-based flow control, not "drop and retransmit." A frozen receiver back-pressures, never drops a packet.
3. **Fat-tree topology** — IB clusters wire up as **non-blocking fat-trees**: every leaf has full bandwidth to every other leaf, even at thousands of nodes.

**Hardware terminology** you'll meet:
- **HCA (Host Channel Adapter)** — the IB NIC. Mellanox / NVIDIA ConnectX-7 is the H100-era card.
- **Subnet manager (`opensm`)** — discovers and configures the fabric.
- **PKey / GUID** — IB equivalents of VLAN and MAC.

## 6 · RoCE v2 — RDMA over Ethernet

InfiniBand is fast but **expensive** and requires a parallel cable plant. **RoCE v2** (RDMA over Converged Ethernet) gives you RDMA semantics over **standard 400/800 Gb Ethernet** with a few requirements:

| Requirement | Why |
|---|---|
| **PFC** (Priority Flow Control) | makes Ethernet lossless for the RDMA traffic class |
| **ECN** (Explicit Congestion Notification) + **DCQCN** | congestion control |
| Tuned switch buffers | absorbing micro-bursts without drops |
| Single L3 domain or tight VXLAN config | RDMA traffic should avoid hops |

In practice: **NVIDIA Spectrum-X**, **Cisco Nexus AI-class**, **Arista 7800R3 AI**, and **Broadcom Tomahawk 5** boxes ship "RoCE-ready" reference designs.

### IB vs RoCE — when to pick what

| Criterion | InfiniBand | RoCE v2 |
|---|---|---|
| Raw bandwidth | up to **XDR 800 Gb/s** | up to **800 Gb/s** Ethernet |
| Latency | best-in-class (sub-µs) | ~1.5-3× IB |
| Cost (per port + cable) | $$$ | $$ |
| Ops complexity | parallel fabric to manage | shares the IT team's Ethernet skill |
| Hyperscalers using it | OpenAI / Anthropic / Microsoft (some), Meta (RSC1) | Meta (RSC2), Azure ND-series, Oracle OCI |
| Multi-tenant | mature (PKeys) | newer; needs careful PFC scoping |

**Default in 2026.** New small-to-mid clusters (< 1 000 GPUs) → **RoCE v2** for cost. Large frontier clusters → **InfiniBand NDR/XDR** for absolute performance — though Meta's RSC2 (24K H100s on Ethernet) is the proof that RoCE can scale.

## 7 · GPUDirect RDMA — the trick that makes the rest work

Without GPUDirect, sending a tensor from GPU 0 on node A to GPU 0 on node B looks like:

```
   GPU 0 (A)  →  pinned host RAM  →  NIC  →  network  →  NIC  →  pinned host RAM  →  GPU 0 (B)
       (1 copy)        (1 copy)             (transfer)       (1 copy)        (1 copy)
```

Four copies, CPU involved at both ends, bus traffic everywhere.

With **GPUDirect RDMA**, the NIC reads directly from GPU memory:

```
   GPU 0 (A)  ──────────────►  NIC  →  network  →  NIC  ──────────────►  GPU 0 (B)
                                              (transfer)
```

Zero CPU involvement, **wire-speed**. This is **mandatory** for any cluster that wants to keep up with H100-class compute.

Three driver/library pieces you must verify are installed:
- **`nv_peer_mem`** (or its successor `nvidia-peermem`) — kernel module that lets the NIC see GPU pages.
- **MOFED** (Mellanox OFED) or **DOCA-OFED** — userspace + drivers for ConnectX cards.
- **NCCL ≥ 2.16** with `--with-cuda` + `--with-mlx5` build.

```bash
# Verify
$ modinfo nvidia-peermem
$ ibv_devinfo                 # should list mlx5_0 etc.
$ ucx_info -d | grep -i cuda  # UCX should report CUDA support
```

If GPUDirect isn't working, NCCL silently falls back to host-staging → ~3× slowdown — easy to miss until profiling.

## 8 · Rail-optimised topology — how DGX racks are wired

The trick that makes 8-GPU servers scale into 1024-GPU pods.

```
                              ┌──── Spine switches (IB or RoCE) ────┐
                              │                                    │
                          ─────────────────── fat-tree ──────────────
                              │     │     │     │     │     │
                            Leaf  Leaf  Leaf  Leaf  Leaf  Leaf  ...
                              │     │     │     │     │     │
                            ─────────────────────────────────── per-rail
                              │     │     │     │     │     │
                    ┌─────────┴─────┴─────┴─────┴─────┴─────┴────────┐
                    │  Node 0:  GPU0─NIC0  GPU1─NIC1  GPU2─NIC2  ... │
                    │  Node 1:  GPU0─NIC0  GPU1─NIC1  GPU2─NIC2  ... │
                    │  ...                                            │
                    └────────────────────────────────────────────────┘
```

**The rule.** GPU `i` in *every* node connects to the **same rail** of the fabric. So GPU 0 in node A talks to GPU 0 in node N over **the shortest path**, regardless of N. AllReduce traffic between matching ranks never crosses the spine more than necessary.

NVIDIA's reference designs that bake this in: **DGX H100 / H200 / GB200 NVL72**, **HGX H100/H200/B100/B200**, **SuperPOD**. Cloud equivalents: **AWS p5 / p6**, **GCP A3 / A3-mega / A3 Ultra**, **Azure ND H100 v5**.

## 9 · Tuning NCCL — env vars + benchmarks

### Env vars you'll set in production

```bash
# Tell NCCL what to use
export NCCL_DEBUG=INFO               # print topology + algorithm choices once at startup
export NCCL_DEBUG_SUBSYS=ALL         # very verbose (use sparingly)
export NCCL_SOCKET_IFNAME=eth0,eth1  # OOB control (NOT data) network interface
export NCCL_IB_HCA=mlx5_0,mlx5_1     # which IB HCAs to use; comma-separated
export NCCL_IB_GID_INDEX=3           # RoCE: the RoCE v2 IPv6 GID; check `show_gids`
export NCCL_NET_GDR_LEVEL=PHB        # require GPUDirect RDMA up to this distance
export NCCL_NVLS_ENABLE=1            # use NVSwitch multicast on H100/H200/B200
export NCCL_P2P_DISABLE=0            # set =1 only to debug
export NCCL_IB_TIMEOUT=22            # raise on flaky fabrics
export NCCL_ASYNC_ERROR_HANDLING=1   # don't hang forever on partial failures
```

### `nccl-tests` — the canonical benchmark

```bash
$ git clone https://github.com/NVIDIA/nccl-tests
$ cd nccl-tests && make MPI=1 NCCL_HOME=/usr CUDA_HOME=/usr/local/cuda
$ mpirun -np 16 -hostfile hostfile \
    ./build/all_reduce_perf -b 8 -e 8G -f 2 -g 1

   #  size         time     algbw   busbw
   8           5.43 us    1.47 GB/s    1.47 GB/s
   16          5.40 us    2.96 GB/s    2.96 GB/s
   ...
   8G          340000 us  24.7 GB/s   42.2 GB/s    ← what you actually want to see
```

Read the **`busbw`** column — it's "effective bandwidth across the bus per GPU." For 8× H100 + NVSwitch you should see ~330+ GB/s on large messages. Across nodes over NDR IB you should see ~40+ GB/s. If the numbers are an order of magnitude off, **stop training, fix networking first**.

### A diagnostic checklist
1. `nvidia-smi topo -m` — does every GPU see NV-something to every other? (If `SYS` shows up: PCIe NUMA misalignment.)
2. `ibstat` / `mlxconfig -d ... query` — are HCAs up, at the speed you paid for?
3. `nccl-tests/all_reduce_perf` — does intra-node match the NVSwitch spec?
4. Add a second node: does inter-node `busbw` get to ~80 % of NIC line rate?
5. Profile a training step with **`nsys`** or **`nsight-systems`** — verify NCCL collectives overlap with compute.

## 10 · The 2025-26 fabric landscape

```
                            INTRA-NODE                       INTER-NODE
                       ────────────────────             ────────────────────
   2020   A100         NVLink 3      600 GB/s/GPU       HDR    200 Gb/s
   2022   H100         NVLink 4      900 GB/s/GPU       NDR    400 Gb/s
   2024   H200         NVLink 4      900 GB/s/GPU       NDR    400 Gb/s
   2025   B100/B200    NVLink 5    1.8 TB/s/GPU         XDR    800 Gb/s
   2025+  GB200 NVL72  NVLink 5    +130 TB/s aggregate  XDR    800 Gb/s

   Ethernet side (RoCE v2):
   2024   Spectrum-X / Tomahawk 5     400 / 800 GbE   AI-class lossless ETh
   2025+  Spectrum-X "Bluefield-3"    800 GbE         CPU offload, telemetry
```

### What's new in 2025-26

- **GB200 NVL72** is the first system where the *boundary* between intra-node and inter-node blurs: 72 Blackwell GPUs are linked into one NVLink domain via copper and an NVLink switch tray. From software's view, they're one big "node."
- **NDR IB → XDR IB** doubles inter-node bandwidth on the same cable plant when you swap HCAs + switches.
- **Spectrum-X** (NVIDIA's RoCE switch line) closes the gap with IB for clusters under ~10 000 GPUs. Many hyperscalers (Meta RSC2, Azure ND-series) chose this over IB.
- **CXL 2.0/3.0** is the next-gen cache-coherent bus — interesting for **memory pooling** (a future module). Not yet on the GPU path in volume.

### Honest field rules
- Sub-1 000 GPUs and you mostly fit in a few nodes → **NVLink intra-node + 100/200 Gb RoCE** is enough.
- 1 000-10 000 GPUs frontier-LLM training → **NVLink intra + NDR or XDR IB inter**, rail-optimised.
- > 10 000 GPUs → custom topologies. Meta's RSC2 (Ethernet) and the post-Frontier supercomputers are reference points.

## ✅ Recap
- "All-reduce" is a stack: **NCCL → CUDA IPC → NVLink in-node → RDMA over IB/RoCE between nodes**.
- **NCCL** picks the algorithm; you provide the topology + env vars + GPUDirect.
- **NVLink + NVSwitch** make an 8-GPU node a bandwidth-rich mesh. Outside the box you need IB or RoCE.
- **InfiniBand NDR/XDR** is the gold standard; **RoCE v2** on Spectrum-X/Tomahawk-5 is the cost-effective Ethernet equivalent.
- **GPUDirect RDMA** is **mandatory** — without it NCCL falls back to host-staging and you lose ~3× throughput silently.
- **Rail-optimised fat-trees** are what keeps thousand-GPU clusters scaling.
- Tune with `NCCL_DEBUG=INFO` + `nccl-tests` + `nvidia-smi topo -m`. Fix networking before model code.

Next: **M78 — Bare-Metal Provisioning + Parallel Storage**.
